# Homework 3

Viraj Dua  
GitHub: VirajD24  
DSCI 552: Machine Learning

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Keeping table values easy to read.
pd.set_option("display.precision", 4)

DATA_DIR = Path("AReM")
SERIES = ["avg_rss12", "var_rss12", "avg_rss13",
          "var_rss13", "avg_rss23", "var_rss23"]
STATISTICS = ["min", "max", "mean", "median", "std", "q1", "q3"]

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Load the data

I use the given AReM data.

The loader handles minor formatting differences in the data files. Most recordings have 480 readings. `sitting/dataset8.csv` has 479, so I calculate its features from the available readings.

In [2]:
files = sorted(
    DATA_DIR.glob("*/*.csv"),
    key=lambda path: (path.parent.name, int(path.stem.removeprefix("dataset"))),
)

recordings = []
records = []

for path in files:
    data = pd.read_csv(
        path,
        comment="#",
        header=None,
        names=["time"] + SERIES,
        sep=r",|\s+",
        engine="python",
        usecols=range(7),
    )
    recordings.append(data[SERIES])
    records.append({
        "Activity": path.parent.name,
        "Dataset": int(path.stem.removeprefix("dataset")),
        "Readings": len(data),
    })

metadata = pd.DataFrame(records, index=range(1, len(records) + 1))
metadata.index.name = "Instance"

print(f"Activities: {metadata['Activity'].nunique()}")
print(f"Recordings: {len(recordings)}")
print("Readings per recording:", metadata["Readings"].value_counts().to_dict())
display(metadata.head())

Activities: 7
Recordings: 88
Readings per recording: {480: 87, 479: 1}


,Activity,Dataset,Readings
Instance,,,
1,bending1,1,480
2,bending1,2,480
3,bending1,3,480
4,bending1,4,480
5,bending1,5,480


### (b) Training and test data

Datasets 1 and 2 from `bending1` and `bending2` are test data. For every other activity, datasets 1, 2, and 3 are test data. All remaining recordings are training data.

In [3]:
is_bending = metadata["Activity"].isin(["bending1", "bending2"])
is_test = (is_bending & (metadata["Dataset"] <= 2)) | (
    ~is_bending & (metadata["Dataset"] <= 3)
)
metadata["Split"] = np.where(is_test, "Test", "Train")

split_counts = pd.crosstab(metadata["Activity"], metadata["Split"])
display(split_counts[["Train", "Test"]])
print(f"Total: {(~is_test).sum()} training recordings and {is_test.sum()} test recordings")

Split,Train,Test
Activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3


Total: 69 training recordings and 19 test recordings


### (c) Feature extraction

#### i. Common time-domain features

Time-domain features summarize the readings directly or how they change over time. Examples include:

| What they describe | Examples |
|---|---|
| Typical values and extremes | Mean, median, minimum, maximum, quartiles |
| Spread | Standard deviation, variance, range, interquartile range |
| Distribution shape | Skewness, kurtosis |
| Signal size | Energy, root mean square |
| Changes over time | Average absolute change, trend, peak count, autocorrelation |

Source: [tsfresh feature documentation](https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html).

#### ii. Extract the seven required statistics

For each of the six time series, I calculate minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile. This produces 42 features per recording. I use the original values without scaling.

The suffix identifies the time series in the order listed in `SERIES`. For example, `mean_1` is the mean of `avg_rss12`, and `q3_6` is the third quartile of `var_rss23`. Standard deviations use `ddof=1` (sample standard deviation).

In [4]:
feature_rows = []

for recording in recordings:
    stats = recording.describe().loc[["min", "max", "mean", "50%", "std", "25%", "75%"]]
    feature_rows.append(stats.T.to_numpy().ravel())

feature_names = [f"{stat}_{i}" for i in range(1, 7) for stat in STATISTICS]
features = pd.DataFrame(feature_rows, columns=feature_names, index=metadata.index)

train_features = features.loc[metadata["Split"] == "Train"]
test_features = features.loc[metadata["Split"] == "Test"]

print(f"Feature table: {features.shape[0]} recordings x {features.shape[1]} features")
print("Training features:", train_features.shape)
print("Test features:", test_features.shape)

# Preview the seven features for the first time series.
display(features.iloc[:5, :7])

Feature table: 88 recordings x 42 features
Training features: (69, 42)
Test features: (19, 42)


,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1
Instance,,,,,,,
1,37.25,45.00,40.6248,40.50,1.4770,39.25,42.00
2,38.00,45.67,42.8128,42.50,1.4355,42.00,43.67
3,35.00,47.40,43.9545,44.33,1.5588,43.00,45.00
4,33.00,47.75,42.1798,43.50,3.6707,39.15,45.00
5,33.00,45.75,41.6781,41.75,2.2435,41.33,42.75


#### iii. Standard deviations and 90% bootstrap confidence intervals

I estimate the standard deviation of each feature across all 88 recordings. This is different from the `std` feature above, which measures variation within one time series.

For each of 10,000 bootstrap samples, I draw 88 recordings with replacement and calculate the sample standard deviation of every feature. The 5th and 95th percentiles give its 90% confidence interval. Also, the fixed seed makes my results reproducible.

In [5]:
rng = np.random.default_rng(42)
sample_indices = rng.integers(0, len(features), size=(10000, len(features)))
confidence_rows = []

for column in features:
    values = features[column].to_numpy()
    bootstrap_std = values[sample_indices].std(axis=1, ddof=1)
    lower, upper = np.percentile(bootstrap_std, [5, 95])
    confidence_rows.append({
        "Feature": column,
        "Standard deviation": values.std(ddof=1),
        "90% CI lower": lower,
        "90% CI upper": upper,
    })

confidence_intervals = pd.DataFrame(confidence_rows).set_index("Feature")
display(confidence_intervals.round(4))

,Standard deviation,90% CI lower,90% CI upper
Feature,,,
min_1,9.5700,8.2360,10.7739
max_1,4.3944,3.3399,5.2931
mean_1,5.3357,4.7038,5.8758
median_1,5.4401,4.7840,6.0063
std_1,1.7722,1.5694,1.9495
q1_1,6.1536,5.5738,6.6486
q3_1,5.1389,4.3386,5.8340
min_2,0.0000,0.0000,0.0000
max_2,5.0627,4.6306,5.4011


#### iv. Choose three important feature types

I would choose **mean, maximum, and standard deviation**:

- **Mean:** describes the average signal level during an activity.
- **Maximum:** captures the strongest signal observed during the recording.
- **Standard deviation:** measures how much the signal varies around the mean.

These describe the signal's level, peaks, and variation; arguably the most important things.

## 2. ISLR 3.7.4

RSS is the sum of squared differences between observed and predicted responses (basically summing the squared residuals). The cubic model includes the linear model as a special case, with the squared and cubic coefficients set to zero.

### (a) True relationship is linear: training RSS

The cubic model's training RSS is less than or equal to the linear model's. Its extra terms give it more freedom to fit the training data, and it can always reproduce the linear fit by setting those terms to zero.

### (b) True relationship is linear: test RSS

I would generally expect the linear model to have lower test RSS. The extra cubic terms can fit noise in the training data without capturing any additional true pattern. This is an expectation, not a guarantee for every test set.

### (c) True relationship is nonlinear: training RSS

The cubic model still has training RSS less than or equal to the linear model's. The same argument from part (a) holds regardless of the true relationship.

### (d) True relationship is nonlinear: test RSS

There is not enough information to decide. The cubic model may capture the curve better, but its extra terms can also fit noise. Which model has lower test RSS depends on the shape of the relationship and how much noise is present.